In [1]:
import pandas as pd
from pathlib import Path

DATA = Path("..") / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)   # raw is never created by code — it's where downloads live

df = pd.read_csv(
    RAW / "mrds-GA.csv",
    low_memory=False,
    encoding="latin-1",
)

print(df.shape)
print(df.columns.tolist())

keep = ["dep_id", "site_name", "latitude", "longitude", "commod1"]
df = df[keep].dropna(subset=["latitude", "longitude"])

print(df["commod1"].value_counts().head(20))

df.head(10)


(2657, 46)
['dep_id', 'url', 'mrds_id', 'mas_id', 'site_name', 'latitude', 'longitude', 'region', 'country', 'state', 'county', 'com_type', 'commod1', 'commod2', 'commod3', 'oper_type', 'dep_type', 'prod_size', 'dev_stat', 'ore', 'gangue', 'other_matl', 'orebody_fm', 'work_type', 'model', 'alteration', 'conc_proc', 'names', 'ore_ctrl', 'reporter', 'hrock_unit', 'hrock_type', 'arock_unit', 'arock_type', 'structure', 'tectonic', 'ref', 'yfp_ba', 'yr_fst_prd', 'ylp_ba', 'yr_lst_prd', 'dy_ba', 'disc_yr', 'prod_yrs', 'discr', 'score']
commod1
Gold                             477
Iron                             325
Kaolin                           249
Mica                             249
Aluminum                         226
Sand and Gravel, Construction    179
Stone, Crushed/Broken            133
Manganese                         92
Clay                              76
Granite                           69
Barium-Barite                     68
Sulfur                            28
Marble, Dime

,dep_id,site_name,latitude,longitude,commod1
0,10022464,Andersonville,32.65534,-83.44779,Aluminum
1,10024974,Cumberland Island Deposit,30.85021,-81.43313,"Thorium, REE, Zirconium, Titanium"
2,10025003,Brunswick Deposit,30.85021,-81.46653,"Titanium, Zirconium"
4,10025912,Laurel Creek Olivine Deposit,34.94591,-83.17548,"Olivine, Vermiculite, Asbestos"
5,10025913,Burton Lake Olivine Deposit,34.87372,-83.57459,Olivine
6,10055267,Gum Creek Pit,32.99046,-83.03738,Kaolin
7,10055268,Stevens Pottery Pit,32.99046,-83.31099,"Sand and Gravel, Construction"
8,10055269,Milledgeville Pit,33.01686,-82.30406,"Sand and Gravel, Construction"
9,10055270,Saxon Pit,33.08766,-82.01655,"Sand and Gravel, Construction"
10,10055271,Campania Pit,33.41685,-82.29985,Brick Clay


In [2]:
import requests
import time
import pandas as pd

MACROSTRAT_URL = "https://macrostrat.org/api/geologic_units/map"

def get_geology(lat: float, lng: float) -> dict:
    """Query Macrostrat for the geologic unit at a point.
    Prefers the most detailed (largest-scale) map available."""
    try:
        r = requests.get(
            MACROSTRAT_URL,
            params={"lat": lat, "lng": lng, "scale": "large"},
            timeout=10,
        )
        r.raise_for_status()
        data = r.json()["success"]["data"]

        # fallback
        if not data:
            r = requests.get(
                MACROSTRAT_URL,
                params={"lat": lat, "lng": lng},
                timeout=10,
            )
            r.raise_for_status()
            data = r.json()["success"]["data"]

        if not data:
            return {}  # no map coverage

        # want smallest span, Decision #2
        best = min(data, key=lambda u: (u["b_age"] or 4600) - (u["t_age"] or 0))

        return {
            "lith": best.get("lith"),
            "b_age": best.get("b_age"),
            "t_age": best.get("t_age"),
            "unit_name": best.get("name"),
            "map_source": best.get("source_id"),
        }
    except Exception as e:
        print(f"  miss at ({lat:.3f}, {lng:.3f}): {e}")
        return {}

sample = df.sample(25, random_state=42).reset_index(drop=True)

rows = []
for i, rec in sample.iterrows():
    geo = get_geology(rec["latitude"], rec["longitude"])
    rows.append({
        "dep_id": rec["dep_id"],
        "site_name": rec["site_name"],
        "commodity": rec["commod1"],
        "lat": rec["latitude"],
        "lng": rec["longitude"],
        **geo,
    })
    print(f"{i+1:>2}/25  {str(rec['site_name'])[:35]:<35} -> {geo.get('unit_name', 'NO DATA')}")
    time.sleep(0.5)

day_one = pd.DataFrame(rows)
day_one.to_csv(PROCESSED / "day_one_table.csv", index=False)
day_one

 1/25  Kelly O'Neil Prospects              -> Paleozoic sedimentary rocks
 2/25  Penland Hill                        -> Knox Group undifferentiated
 3/25  H.E.Lucas Property                  -> Knox Group undifferentiated
 4/25  Crosby No. 3 Mine                   -> Late Cretaceous sedimentary
 5/25  Stovall Pit                         -> Neoproterozoic-Cambrian volcanic: interlayered sedimentary and volcanic rocks
 6/25  Tallapoosa Mine                     -> Paleozoic sedimentary rocks
 7/25  Prater Mine                         -> Neoproterozoic-Cambrian sedimentary
 8/25  Huber Mill                          -> Twiggs Clay
 9/25  Young Deer Creek                    -> Mesoproterozoic crystalline metamorphic rocks
10/25  Lot 826                             -> Rome Formation
11/25  Clay Preparation Plant              -> Late Eocene sedimentary
12/25  Lots 191 and 242                    -> Chilhowee Formation
13/25  Draketown Dist. (Douglas Mn Prospec -> Neoproterozoic-Cambrian sedimen

,dep_id,site_name,commodity,lat,lng,lith,b_age,t_age,unit_name,map_source
0,10144000,Kelly O'Neil Prospects,Mica,32.97706,-84.13271,sedimentary rocks,541.0000,419.2000,Paleozoic sedimentary rocks,154
1,10067858,Penland Hill,Manganese,34.03344,-85.28324,"Major:{dolostone}, Minor:{limestone}",493.5750,470.0000,Knox Group undifferentiated,133
2,10216187,H.E.Lucas Property,NaN,34.00074,-85.16934,"Major:{dolostone}, Minor:{limestone}",493.5750,470.0000,Knox Group undifferentiated,133
3,10216302,Crosby No. 3 Mine,Kaolin,32.83296,-83.41069,"sand, chalk, mud",74.9750,71.0833,Late Cretaceous sedimentary,7
4,10134673,Stovall Pit,"Sand and Gravel, Construction",33.67824,-84.61322,volcanic: interlayered sedimentary and volcani...,1000.0000,485.4000,Neoproterozoic-Cambrian volcanic: interlayered...,7
5,10119380,Tallapoosa Mine,Copper,33.85014,-85.08324,sedimentary rocks,541.0000,251.9020,Paleozoic sedimentary rocks,154
6,10305990,Prater Mine,Gold,34.37513,-83.87490,sedimentary,1000.0000,485.4000,Neoproterozoic-Cambrian sedimentary,7
7,10143770,Huber Mill,Kaolin,32.71597,-83.52909,Major:{clay},35.4600,34.6800,Twiggs Clay,133
8,10083725,Young Deer Creek,Gold,34.19183,-84.05821,paragneiss; paragneiss/metavolcanic gneiss,1600.0000,1000.0000,Mesoproterozoic crystalline metamorphic rocks,154
9,10118701,Lot 826,NaN,34.13573,-84.73553,"Major:{sandstone,shale}",514.7500,511.0000,Rome Formation,133
